# Monthly USDT Symbols Data Processor

This notebook processes cryptocurrency data for all USDT symbols from 2020-01 to 2025-07,
saving monthly data as pickle files with multi-index (open_time, symbol).

In [1]:
import pandas as pd
import os
import glob
from datetime import datetime, timedelta
import pickle
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Configuration
DATA_PATH = '/home/craz/crypto/crypto-data/future_data_2/'
OUTPUT_PATH = '/home/craz/crypto/crypto-data/pickle_month_cache/'
START_DATE = '2020-01'
END_DATE = '2025-07'

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"Data source: {DATA_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print(f"Date range: {START_DATE} to {END_DATE}")

Data source: /home/craz/crypto/crypto-data/future_data_2/
Output path: /home/craz/crypto/crypto-data/pickle_month_cache/
Date range: 2020-01 to 2025-07


In [3]:
def get_usdt_symbols():
    """Get all USDT symbol directories"""
    all_dirs = [d for d in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, d))]
    usdt_symbols = [d for d in all_dirs if d.endswith('USDT')]
    return sorted(usdt_symbols)

usdt_symbols = get_usdt_symbols()
print(f"Found {len(usdt_symbols)} USDT symbols")
print(f"First 10 symbols: {usdt_symbols[:10]}")

Found 512 USDT symbols
First 10 symbols: ['1000000BOBUSDT', '1000000MOGUSDT', '1000BONKUSDT', '1000CATUSDT', '1000CHEEMSUSDT', '1000FLOKIUSDT', '1000LUNCUSDT', '1000PEPEUSDT', '1000RATSUSDT', '1000SATSUSDT']


In [4]:
def generate_month_list(start_date, end_date):
    """Generate list of year-month strings between start and end dates"""
    start = datetime.strptime(start_date + '-01', '%Y-%m-%d')
    end = datetime.strptime(end_date + '-01', '%Y-%m-%d')
    
    months = []
    current = start
    while current <= end:
        months.append(current.strftime('%Y-%m'))
        # Move to next month
        if current.month == 12:
            current = current.replace(year=current.year + 1, month=1)
        else:
            current = current.replace(month=current.month + 1)
    return months

month_list = generate_month_list(START_DATE, END_DATE)
print(f"Processing {len(month_list)} months")
print(f"Months: {month_list[:5]} ... {month_list[-5:]}")

Processing 67 months
Months: ['2020-01', '2020-02', '2020-03', '2020-04', '2020-05'] ... ['2025-03', '2025-04', '2025-05', '2025-06', '2025-07']


In [5]:
def load_symbol_month_data(symbol, year_month):
    """Load all CSV files for a symbol in a given month"""
    symbol_path = os.path.join(DATA_PATH, symbol)
    if not os.path.exists(symbol_path):
        return None
    
    # Find all CSV files for the given month
    pattern = os.path.join(symbol_path, f"kline_1m_{year_month}-*.csv")
    files = glob.glob(pattern)
    
    if not files:
        return None
    
    # Load and combine all files
    dataframes = []
    for file in sorted(files):
        try:
            df = pd.read_csv(file)
            df['open_time'] = pd.to_datetime(df['open_time'])
            df['symbol'] = symbol
            dataframes.append(df)
        except Exception as e:
            print(f"Error loading {file}: {e}")
            continue
    
    if not dataframes:
        return None
    
    # Combine all dataframes
    combined_df = pd.concat(dataframes, ignore_index=True)
    combined_df = combined_df.sort_values('open_time').reset_index(drop=True)
    
    return combined_df

In [6]:
def process_monthly_data(year_month):
    """Process all USDT symbols for a given month and save as pickle"""
    print(f"\nProcessing {year_month}...")
    
    all_data = []
    successful_symbols = 0
    
    # Progress bar for symbols
    for symbol in tqdm(usdt_symbols, desc=f"Loading {year_month}"):
        data = load_symbol_month_data(symbol, year_month)
        if data is not None:
            all_data.append(data)
            successful_symbols += 1
    
    if not all_data:
        print(f"No data found for {year_month}")
        return
    
    # Combine all symbols data
    combined_df = pd.concat(all_data, ignore_index=True)
    
    # Set multi-index: open_time first, then symbol
    combined_df = combined_df.set_index(['open_time', 'symbol'])
    combined_df = combined_df.sort_index()
    
    # Save as pickle file
    output_file = os.path.join(OUTPUT_PATH, f"usdt_data_{year_month}.pkl")
    with open(output_file, 'wb') as f:
        pickle.dump(combined_df, f)
    
    print(f"Saved {year_month}: {len(combined_df):,} rows, {successful_symbols} symbols -> {output_file}")
    return len(combined_df), successful_symbols

In [7]:
# Test with one month first
test_month = '2020-01'
print(f"Testing with {test_month}...")
rows, symbols = process_monthly_data(test_month)
print(f"Test completed: {rows:,} rows, {symbols} symbols")

Testing with 2020-01...

Processing 2020-01...


Loading 2020-01: 100%|██████████| 512/512 [00:01<00:00, 429.64it/s] 

Saved 2020-01: 285,560 rows, 10 symbols -> /home/craz/crypto/crypto-data/pickle_month_cache/usdt_data_2020-01.pkl
Test completed: 285,560 rows, 10 symbols


In [8]:
# Load and verify the test file
test_file = os.path.join(OUTPUT_PATH, f"usdt_data_{test_month}.pkl")
if os.path.exists(test_file):
    with open(test_file, 'rb') as f:
        test_df = pickle.load(f)
    
    print(f"Test file loaded successfully:")
    print(f"Shape: {test_df.shape}")
    print(f"Index levels: {test_df.index.names}")
    print(f"Columns: {test_df.columns.tolist()}")
    print(f"Date range: {test_df.index.get_level_values('open_time').min()} to {test_df.index.get_level_values('open_time').max()}")
    print(f"Unique symbols: {len(test_df.index.get_level_values('symbol').unique())}")
    print("\nFirst few rows:")
    print(test_df.head())
else:
    print(f"Test file not found: {test_file}")

Test file loaded successfully:
Shape: (285560, 5)
Index levels: ['open_time', 'symbol']
Columns: ['open', 'high', 'low', 'close', 'volume']
Date range: 2020-01-01 00:00:00 to 2020-01-31 23:59:00
Unique symbols: 10

First few rows:
                                open     high      low    close    volume
open_time           symbol                                               
2020-01-01 00:00:00 BCHUSDT   206.58   206.85   206.15   206.85   451.493
                    BTCUSDT  7191.80  7198.95  7184.46  7198.35   434.949
                    ETHUSDT   129.61   129.75   129.06   129.70  1595.727
2020-01-01 00:01:00 BCHUSDT   206.85   207.73   206.85   207.67   676.089
                    BTCUSDT  7198.14  7216.85  7197.87  7212.94   397.023


In [9]:
# Process all months
print(f"Starting batch processing for all {len(month_list)} months...")
print("This may take a while...\n")

total_rows = 0
total_files = 0
failed_months = []

for month in month_list:
    try:
        result = process_monthly_data(month)
        if result:
            rows, symbols = result
            total_rows += rows
            total_files += 1
        else:
            failed_months.append(month)
    except Exception as e:
        print(f"Error processing {month}: {e}")
        failed_months.append(month)

print(f"\n=== BATCH PROCESSING COMPLETE ===")
print(f"Successfully processed: {total_files} months")
print(f"Total data rows: {total_rows:,}")
if failed_months:
    print(f"Failed months: {failed_months}")
else:
    print("All months processed successfully!")

Starting batch processing for all 67 months...
This may take a while...


Processing 2020-01...


Loading 2020-01: 100%|██████████| 512/512 [00:00<00:00, 1065.74it/s]


Saved 2020-01: 285,560 rows, 10 symbols -> /home/craz/crypto/crypto-data/pickle_month_cache/usdt_data_2020-01.pkl

Processing 2020-02...


Loading 2020-02: 100%|██████████| 512/512 [00:01<00:00, 421.28it/s]


Saved 2020-02: 760,470 rows, 23 symbols -> /home/craz/crypto/crypto-data/pickle_month_cache/usdt_data_2020-02.pkl

Processing 2020-03...


...

Saved 2025-05: 20,088,389 rows, 461 symbols -> /home/craz/crypto/crypto-data/pickle_month_cache/usdt_data_2025-05.pkl

Processing 2025-06...


Loading 2025-06: 100%|██████████| 512/512 [00:29<00:00, 17.33it/s]


Saved 2025-06: 20,371,515 rows, 480 symbols -> /home/craz/crypto/crypto-data/pickle_month_cache/usdt_data_2025-06.pkl

Processing 2025-07...


Loading 2025-07: 100%|██████████| 512/512 [00:31<00:00, 16.15it/s]


Saved 2025-07: 21,863,880 rows, 503 symbols -> /home/craz/crypto/crypto-data/pickle_month_cache/usdt_data_2025-07.pkl

=== BATCH PROCESSING COMPLETE ===
Successfully processed: 67 months
Total data rows: 511,201,470
All months processed successfully!


In [10]:
# List all generated pickle files
pickle_files = glob.glob(os.path.join(OUTPUT_PATH, "usdt_data_*.pkl"))
pickle_files.sort()

print(f"Generated {len(pickle_files)} pickle files:")
for i, file in enumerate(pickle_files):
    file_size = os.path.getsize(file) / (1024*1024)  # MB
    file_name = os.path.basename(file)
    print(f"{i+1:2d}. {file_name} ({file_size:.1f} MB)")

Generated 67 pickle files:
 1. usdt_data_2020-01.pkl (12.6 MB)
 2. usdt_data_2020-02.pkl (33.0 MB)
 3. usdt_data_2020-03.pkl (44.4 MB)
 4. usdt_data_2020-04.pkl (43.0 MB)
 5. usdt_data_2020-05.pkl (44.7 MB)
 6. usdt_data_2020-06.pkl (47.4 MB)
 7. usdt_data_2020-07.pkl (60.0 MB)
 8. usdt_data_2020-08.pkl (71.6 MB)
 9. usdt_data_2020-09.pkl (91.4 MB)
10. usdt_data_2020-10.pkl (114.3 MB)
11. usdt_data_2020-11.pkl (121.1 MB)
12. usdt_data_2020-12.pkl (133.2 MB)
13. usdt_data_2021-01.pkl (137.7 MB)
14. usdt_data_2021-02.pkl (130.6 MB)
15. usdt_data_2021-03.pkl (161.7 MB)
16. usdt_data_2021-04.pkl (175.8 MB)
17. usdt_data_2021-05.pkl (186.3 MB)
18. usdt_data_2021-06.pkl (183.8 MB)
19. usdt_data_2021-07.pkl (191.9 MB)
20. usdt_data_2021-08.pkl (194.6 MB)
21. usdt_data_2021-09.pkl (198.2 MB)
22. usdt_data_2021-10.pkl (213.4 MB)
23. usdt_data_2021-11.pkl (211.1 MB)
24. usdt_data_2021-12.pkl (221.1 MB)
25. usdt_data_2022-01.pkl (226.0 MB)
26. usdt_data_2022-02.pkl (207.1 MB)
27. usdt_data_2022-0

In [11]:
# Summary statistics
print("\n=== SUMMARY STATISTICS ===")
total_size_mb = sum(os.path.getsize(f) for f in pickle_files) / (1024*1024)
print(f"Total pickle files: {len(pickle_files)}")
print(f"Total size: {total_size_mb:.1f} MB")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"USDT symbols: {len(usdt_symbols)}")
print(f"Output directory: {OUTPUT_PATH}")

if pickle_files:
    print(f"\nFirst file: {os.path.basename(pickle_files[0])}")
    print(f"Last file: {os.path.basename(pickle_files[-1])}")
    
print("\nProcessing completed successfully! 🎉")


=== SUMMARY STATISTICS ===
Total pickle files: 67
Total size: 22366.7 MB
Date range: 2020-01 to 2025-07
USDT symbols: 512
Output directory: /home/craz/crypto/crypto-data/pickle_month_cache/

First file: usdt_data_2020-01.pkl
Last file: usdt_data_2025-07.pkl

Processing completed successfully! 🎉
